# Coarse Fog Masking 實作與測試 (真實影片版)

這份 Notebook 實作了專案 Proposal 中定義的 **Coarse Fog Masking (粗略遮罩)**。利用局部對比度、HSV 飽和度以及影像梯度，透過聯集投票機制來找出初步的起霧像素。

**成員 A 的任務**：請將真實的起霧影片放入資料夾中，修改下方 Cell 的 `video_path`，並不斷微調 `thresh_var`, `thresh_sat`, `thresh_grad` 這三個參數，直到遮罩能完美捕捉到水氣且不誤判正常場景為止。

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

# 設定 matplotlib 顯示大小
plt.rcParams['figure.figsize'] = [18, 12]

part 1: coarse fog mask

### 核心函式定義

In [2]:
def get_coarse_fog_mask(image_bgr, 
                        patch_size=15, 
                        thresh_var=0.015, 
                        thresh_sat=0.1, 
                        thresh_grad=0.15,
                        thresh_val=0.5):  # 🌟 新增明度閾值參數
    # 1. 影像預處理
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    
    # 特徵一：對比度特徵 (Contrast)
    mean_of_sq = cv2.blur(gray**2, (patch_size, patch_size))
    sq_of_mean = cv2.blur(gray, (patch_size, patch_size))**2
    variance = np.clip(mean_of_sq - sq_of_mean, 0, None)
    mask_contrast = variance < thresh_var
    
    # 特徵二 & 新增的特徵四：飽和度 (Saturation) 與 明度 (Value)
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    sat = hsv[:, :, 1].astype(np.float32) / 255.0
    mask_sat = sat < thresh_sat
    
    val = hsv[:, :, 2].astype(np.float32) / 255.0
    mask_bright = val > thresh_val  # 🌟 新增：亮度必須大於閾值
    
    # 特徵三：梯度特徵 (Gradient)
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(sobelx**2 + sobely**2)
    mask_grad = grad_mag < thresh_grad
    
    # 決策：投票機制 (符合兩項以上)
    votes = mask_contrast.astype(np.uint8) + \
            mask_sat.astype(np.uint8) + \
            mask_grad.astype(np.uint8)
            
    # 🌟 關鍵修改：得票數 >= 2，【且】亮度必須夠高
    coarse_mask = ((votes >= 2) & mask_bright).astype(np.uint8) * 255
    
    # 形態學操作去噪
    kernel = np.ones((5, 5), np.uint8)
    coarse_mask = cv2.morphologyEx(coarse_mask, cv2.MORPH_OPEN, kernel)
    coarse_mask = cv2.morphologyEx(coarse_mask, cv2.MORPH_CLOSE, kernel)
    
    # 🌟 更新 debug 字典，加入 brightness
    debug_masks = {
        'contrast': (mask_contrast.astype(np.uint8) * 255),
        'saturation': (mask_sat.astype(np.uint8) * 255),
        'gradient': (mask_grad.astype(np.uint8) * 255),
        'brightness': (mask_bright.astype(np.uint8) * 255), # <--- 這裡加進去
        'votes_map': (votes * 85).astype(np.uint8)
    }
    
    return coarse_mask, debug_masks

### 讀取真實影片與視覺化
請將 `video_path` 替換為你們手邊的影片路徑。你可以透過改變 `frame_id` 來隨機抽查影片中的不同時間點，確保參數在整支影片中都具備泛化能力。

part 2: homology transformation

In [3]:
def align_video_sequential(video_path, start_frame, end_frame):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    ret, frame_prev = cap.read()
    if not ret: return None, None, None
        
    h, w = frame_prev.shape[:2]
    
    mask_prev, _ = get_coarse_fog_mask(frame_prev)
    valid_mask_prev = cv2.bitwise_not(mask_prev)
    gray_prev = cv2.cvtColor(frame_prev, cv2.COLOR_BGR2GRAY)
    
    sift = cv2.SIFT_create()
    kp_prev, des_prev = sift.detectAndCompute(gray_prev, mask=valid_mask_prev)
    
    H_accumulated = np.eye(3, dtype=np.float32)
    raw_H_list = [H_accumulated.copy()] 
    
    bf = cv2.BFMatcher()
    
    print(f"🔗 開始連鎖對齊 (純矩陣運算)：第 {start_frame} 到 {end_frame} 幀")
    for i in tqdm(range(start_frame + 1, end_frame + 1)):
        ret, frame_curr = cap.read()
        if not ret: break
            
        mask_curr, _ = get_coarse_fog_mask(frame_curr)
        valid_mask_curr = cv2.bitwise_not(mask_curr)
        gray_curr = cv2.cvtColor(frame_curr, cv2.COLOR_BGR2GRAY)
        kp_curr, des_curr = sift.detectAndCompute(gray_curr, mask=valid_mask_curr)
        
        matches = bf.knnMatch(des_curr, des_prev, k=2)
        good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]
        
        # 預設：假設這幀對齊失敗，沿用上一幀的累積矩陣
        H_curr_to_prev = None 
        
        if len(good_matches) > 15: # 🌟 提高標準，原本是 10
            src_pts = np.float32([kp_curr[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
            dst_pts = np.float32([kp_prev[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
            
            # 使用更嚴格的 RANSAC 閾值
            affine_matrix, mask = cv2.estimateAffinePartial2D(src_pts, dst_pts, method=cv2.RANSAC, ransacReprojThreshold=3.0)
            
            # ==========================================
            # 🌟 核心防護網：檢查矩陣是否合理 (Sanity Check)
            # ==========================================
            if affine_matrix is not None:
                # 把算出來的 2x3 矩陣，補上底部變成 3x3 格式，讓後面的程式不用改
                H_temp = np.eye(3, dtype=np.float32)
                H_temp[0:2, :] = affine_matrix
            else:
                H_temp = None
                
            # === 下面的防護網依然保留，但條件可以稍微放寬 ===
            if H_temp is not None:
                # 檢查平移量 (tx, ty) 是否合理
                tx, ty = H_temp[0, 2], H_temp[1, 2]
                if abs(tx) > w * 0.5 or abs(ty) > h * 0.5:
                    print(f"⚠️ 第 {i} 幀：平移量過大 (tx={tx:.1f}, ty={ty:.1f})，拋棄矩陣。")
                    H_temp = None
            
            if H_temp is not None:
                H_curr_to_prev = H_temp
                
        if H_curr_to_prev is not None:
            H_accumulated = np.matmul(H_accumulated, H_curr_to_prev)
            kp_prev, des_prev = kp_curr, des_curr
        else:
            # 如果這幀對齊失敗 (特徵不夠或被防護網擋下)，我們就當作相機沒動 (沿用 H_accumulated)
            # 注意：這裡不更新 kp_prev，讓下一幀繼續嘗試與上一個成功的幀對齊！
            print(f"⚠️ 第 {i} 幀：特徵追蹤丟失，沿用上一個視角。")
                
        raw_H_list.append(H_accumulated.copy())
            
    cap.release()
    
    # ==========================================
    # 階段 B：精確計算大畫布範圍 (加入畫布大小極限保護)
    # ==========================================
    print("📐 正在精確計算大畫布所需尺寸...")
    corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)
    all_corners = []
    
    for H in raw_H_list:
        transformed_corners = cv2.perspectiveTransform(corners, H)
        all_corners.append(transformed_corners)
        
    all_corners = np.concatenate(all_corners, axis=0)
    
    [x_min, y_min] = np.int32(all_corners.min(axis=0).ravel() - 0.5)
    [x_max, y_max] = np.int32(all_corners.max(axis=0).ravel() + 0.5)
    
    pano_w = x_max - x_min
    pano_h = y_max - y_min
    
    # 🌟 最終防線：如果算出來的畫布還是大得離譜 (大於原始寬度的 15 倍)，強制卡死
    if pano_w > w * 15 or pano_h > h * 15:
        print(f"🚨 嚴重警告：計算出的畫布過大 ({pano_w}x{pano_h})！強制縮減以防崩潰。")
        pano_w = min(pano_w, int(w * 15))
        pano_h = min(pano_h, int(h * 15))
        
        # 修正 x_min, y_min 防止越界
        x_min = max(x_min, -int(w * 7.5))
        y_min = max(y_min, -int(h * 7.5))
    
    T_offset = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]], dtype=np.float32)
    
    final_H_list = []
    for H in raw_H_list:
        final_H_list.append(np.matmul(T_offset, H))
        
    print(f"✅ 畫布計算完成！最佳尺寸為: {pano_w} x {pano_h}")
    
    return final_H_list, pano_w, pano_h

part 3: temporal fusion & repainting

In [4]:
def process_dynamic_fusion(video_path, start_frame, H_list, pano_w, pano_h):
    cap = cv2.VideoCapture(video_path)
    
    # ==========================================
    # Pass 1: 智能背景提取 (只收集沒有霧的像素)
    # ==========================================
    print("🔍 Pass 1: 建立全域乾淨背景 (剔除霧霾干擾與邊界污染)...")
    
    pano_bg_sum = np.zeros((pano_h, pano_w, 3), dtype=np.float32)
    pano_weight = np.zeros((pano_h, pano_w, 1), dtype=np.float32)
    
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    for t in tqdm(range(len(H_list))):
        ret, frame = cap.read()
        if not ret: break
            
        # 1. 取得這幀哪裡有霧 (255=霧, 0=乾淨)
        fog_mask, _ = get_coarse_fog_mask(frame)
        
        # 2. 反轉遮罩，變成乾淨權重 (1.0=乾淨可以提取, 0.0=有霧不要提取)
        clean_mask = (255 - fog_mask).astype(np.float32) / 255.0
        
        # 3. 把它們變形到大畫布上 (這就是你剛才不小心刪除掉的 warped_frame!)
        H = H_list[t]
        warped_frame = cv2.warpPerspective(frame, H, (pano_w, pano_h)).astype(np.float32)
        warped_clean_mask = cv2.warpPerspective(clean_mask, H, (pano_w, pano_h))
        warped_clean_mask = np.expand_dims(warped_clean_mask, axis=-1)
        
        # 4. 找出有效像素的 2D 遮罩
        valid_mask_2d = np.any(warped_frame > 0, axis=-1).astype(np.uint8)
        
        # 5. 邊界向內侵蝕 (Erode) 3 次，徹底切除與黑色背景插值產生的污染邊緣
        kernel_erode = np.ones((5, 5), np.uint8)
        valid_mask_2d = cv2.erode(valid_mask_2d, kernel_erode, iterations=3)
        
        # 6. 轉回 3D 浮點數矩陣以利後續相乘
        valid_pixels = np.expand_dims(valid_mask_2d, axis=-1).astype(np.float32)
        
        # 🌟 核心：只有乾淨且非邊緣的像素才會被加進背景統計中！
        pano_bg_sum += warped_frame * warped_clean_mask * valid_pixels
        pano_weight += warped_clean_mask * valid_pixels

    # 計算平均值得到最終背景 (避免除以零)
    clean_bg_ref = np.zeros_like(pano_bg_sum)
    valid_bg = pano_weight > 0
    clean_bg_ref[valid_bg.repeat(3, axis=-1)] = (pano_bg_sum / (pano_weight + 1e-6))[valid_bg.repeat(3, axis=-1)]
    clean_bg_ref = np.clip(clean_bg_ref, 0, 255).astype(np.uint8)
    
    # 儲存這張完美去霧世界圖以供檢查
    cv2.imwrite('smart_clean_bg.jpg', clean_bg_ref)

    # ==========================================
    # Pass 2: 將背景拉回與原始影片融合
    # ==========================================
    print("🎨 Pass 2: 動態視野拉回與無縫融合...")
    final_dehazed_frames = []
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    prev_mask_float = None 
    
    for t in tqdm(range(len(H_list))):
        ret, frame_raw = cap.read()
        if not ret: break
            
        raw_h, raw_w = frame_raw.shape[:2]
        
        ret_inv, H_inv = cv2.invert(H_list[t])
        if not ret_inv: H_inv = np.eye(3)
            
        # 從大背景中切出當前相機視野
        bg_at_current_view = cv2.warpPerspective(clean_bg_ref, H_inv, (raw_w, raw_h))
        
        mask_coarse, _ = get_coarse_fog_mask(frame_raw) 
        
        # 遮罩大範圍高斯模糊 (消除斷層感)
        mask_float = cv2.GaussianBlur(mask_coarse.astype(np.float32)/255.0, (81, 81), 0)
        
        # 時序平滑 (讓水花/車輛誤判造成的閃爍變得平滑)
        if prev_mask_float is None:
            prev_mask_float = mask_float
        else:
            mask_float = 0.7 * prev_mask_float + 0.3 * mask_float 
            prev_mask_float = mask_float
            
        mask_3c = np.stack([mask_float]*3, axis=-1)
        
        # 1. 找出拉回來的背景哪裡是有效的
        valid_bg_2d = np.any(bg_at_current_view > 0, axis=-1).astype(np.uint8)
        
        # 2. 邊界向內侵蝕，避免拉回來時產生的黑邊
        valid_bg_2d = cv2.erode(valid_bg_2d, kernel_erode, iterations=3)
        
        # 3. 邊緣羽化 (Feathering)：讓背景與當下視野的交界處漸層過渡
        valid_bg_float = cv2.GaussianBlur(valid_bg_2d.astype(np.float32), (41, 41), 0)
        valid_bg_mask = np.expand_dims(valid_bg_float, axis=-1)
        
        # 4. 最終遮罩組合：有霧的地方 + 有效且經過羽化的背景區域
        final_mask = mask_3c * valid_bg_mask
        
        # 最終混合！
        blended = frame_raw * (1.0 - final_mask) + bg_at_current_view * final_mask
        blended = np.clip(blended, 0, 255).astype(np.uint8)
        
        final_dehazed_frames.append(blended)
        
    cap.release()
    return final_dehazed_frames

In [7]:
video_path = './data/foggy_vid/6.mp4'

# 自動取得影片總幀數
cap = cv2.VideoCapture(video_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

print(f"🎬 影片總幀數為: {total_frames}")

# 1. 取得矩陣 (超級快，不會佔用記憶體)
matrices, p_w, p_h = align_video_sequential(video_path, 0, total_frames - 1)

# 2. 兩階段提取與融合
dehazed_video = process_dynamic_fusion(video_path, 0, matrices, p_w, p_h)

# 3. 儲存
if dehazed_video:
    frame_h, frame_w = dehazed_video[0].shape[:2]
    out = cv2.VideoWriter('dehazed_output_smart.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 30, (frame_w, frame_h))
    for f in dehazed_video: out.write(f)
    out.release()
    print("✅ 成功！影片已儲存。請同時查看資料夾下的 smart_clean_bg.jpg")

🎬 影片總幀數為: 232
🔗 開始連鎖對齊 (純矩陣運算)：第 0 到 231 幀


100%|██████████| 231/231 [00:13<00:00, 16.84it/s]


📐 正在精確計算大畫布所需尺寸...
✅ 畫布計算完成！最佳尺寸為: 1319 x 872
🔍 Pass 1: 建立全域乾淨背景 (剔除霧霾干擾與邊界污染)...


100%|██████████| 232/232 [00:15<00:00, 15.39it/s]


🎨 Pass 2: 動態視野拉回與無縫融合...


100%|██████████| 232/232 [00:09<00:00, 23.63it/s]


✅ 成功！影片已儲存。請同時查看資料夾下的 smart_clean_bg.jpg
